# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saad5987/ML-/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two Paper Findings + My Methodology Questions

### Finding 1 — The Anatomy of Growing Content

The paper reports that growing content tended to be longer, younger, and slightly better positioned than declining content. The growing group averaged about 3.2K words and 184 days of age, compared with about 2.3K words and 230 days for declining content.

**Methodology question:** Where does the "growing" label come from, and does the comparison establish an association only, or does the evidence support saying that greater content depth or younger age causes growth?

This is a constructive question because the paper describes the analysis as observational. The observed differences are useful for decision support, but they should not automatically be interpreted as causal effects.

### Finding 2 — The Content Performance Curve

The paper reports that content performance peaks around 61–90 days, declines after 270 days, and that the 365+ rebound is concentrated in older pages that were refreshed.

**Methodology question:** How was content age separated from other factors such as refresh history, and does the validation or comparison design support attributing the observed performance pattern specifically to age?

This question is intended to distinguish an observed relationship from a causal claim. The age buckets provide a useful directional comparison, but other factors may also contribute to the differences between groups.

## 2. My Model Under an Honest Split

The Week-5 model used a random 80/20 split and measured 1.0000 accuracy. For a more honest validation check, I will use a time-aware split: earlier observations will be used for training and later observations will be held out for testing.

This provides a before/after comparison and better reflects how the model would be evaluated when predictions are made on future observations. The result will be treated as measured performance under this specific validation design rather than evidence of general real-world performance.

The time-aware check was performed on the same 50,000-row sample used in Week 5, so the observed before/after result covers a limited date range. This is a validation improvement over a random split, but it is not sufficient to establish performance across a broader future period.

In [23]:
con.sql("""
SELECT COUNT(*) AS total_rows
FROM daily
""").df()

,total_rows
0,11694072


In [24]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load the same data used for the Week-5 model
data = con.sql("""
SELECT
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions,
    CASE
        WHEN gsc_impressions >= 1000
        AND (
            100.0 * gsc_clicks /
            NULLIF(gsc_impressions, 0)
        ) < 1
        THEN 1
        ELSE 0
    END AS target
FROM daily
WHERE gsc_data_available IS TRUE
  AND gsc_impressions > 0
LIMIT 50000
""").df()

# Convert date column
data["report_date"] = pd.to_datetime(data["report_date"])

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

# -------------------------
# BEFORE: Week-5 random split
# -------------------------
from sklearn.model_selection import train_test_split

X = data[features]
y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

before_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

before_model.fit(X_train, y_train)

before_pred = before_model.predict(X_test)
before_accuracy = accuracy_score(y_test, before_pred)

# -------------------------
# AFTER: Time-aware split
# -------------------------

data = data.sort_values("report_date").reset_index(drop=True)

split_index = int(len(data) * 0.80)

train = data.iloc[:split_index]
test = data.iloc[split_index:]

after_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

after_model.fit(
    train[features],
    train["target"]
)

after_pred = after_model.predict(test[features])
after_accuracy = accuracy_score(
    test["target"],
    after_pred
)

# -------------------------
# Results
# -------------------------

comparison = pd.DataFrame({
    "Validation": [
        "Week-5 Random 80/20 Split",
        "Time-Aware 80/20 Split"
    ],
    "Accuracy": [
        before_accuracy,
        after_accuracy
    ]
})

print("Before / After Validation Comparison")
display(comparison)

print("\nTime-aware split dates:")
print("Training:", train["report_date"].min(), "to", train["report_date"].max())
print("Testing :", test["report_date"].min(), "to", test["report_date"].max())

print("\nTime-aware Classification Report:")
print(classification_report(
    test["target"],
    after_pred,
    zero_division=0
))

Before / After Validation Comparison


,Validation,Accuracy
0,Week-5 Random 80/20 Split,1.0000
1,Time-Aware 80/20 Split,0.9999



Time-aware split dates:
Training: 2026-06-01 00:00:00 to 2026-06-02 00:00:00
Testing : 2026-06-02 00:00:00 to 2026-06-03 00:00:00

Time-aware Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      9975
           1       0.96      1.00      0.98        25

    accuracy                           1.00     10000
   macro avg       0.98      1.00      0.99     10000
weighted avg       1.00      1.00      1.00     10000



## 3. Leakage Audit

The Week-5 target was constructed using `gsc_impressions` and `gsc_clicks`: a row was labeled positive when impressions were at least 1,000 and the calculated CTR was below 1%.

Both `gsc_impressions` and `gsc_clicks` were also included as model features. Therefore, the model has direct access to variables used to construct the target. This creates a target-feature dependency and can make the measured accuracy look stronger than it would be with an independently defined outcome.

The other Week-5 features (`gsc_avg_position`, `ga4_pageviews`, and `ga4_sessions`) were not directly used in the target formula. However, they should still be treated as signals requiring temporal and feature-availability checks before making broader claims.

**Verdict:** The Week-5 model has a leakage/target-construction concern because two input features directly contribute to the target definition. The 1.0000 accuracy should therefore be treated as measured performance under that setup, not evidence of general predictive performance.

In [25]:
# Audit the Week-5 target construction against the model features

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]

target_dependencies = [
    "gsc_impressions",
    "gsc_clicks"
]

audit = pd.DataFrame({
    "feature": feature_columns,
    "used_in_target_definition": [
        feature in target_dependencies
        for feature in feature_columns
    ]
})

display(audit)

print("Features directly used to construct the target:")
print(target_dependencies)

print("\nLeakage audit verdict:")
print(
    "CONCERN — gsc_impressions and gsc_clicks are model inputs "
    "and were also used to construct the target."
)

,feature,used_in_target_definition
0,gsc_impressions,True
1,gsc_clicks,True
2,gsc_avg_position,False
3,ga4_pageviews,False
4,ga4_sessions,False


Features directly used to construct the target:
['gsc_impressions', 'gsc_clicks']

Leakage audit verdict:
CONCERN — gsc_impressions and gsc_clicks are model inputs and were also used to construct the target.


## 4. Claim Rewrite

### Original claim

The Week-5 Decision Tree model achieved 1.0000 accuracy, showing that the model can predict the target with perfect performance.

### Safer claim

The Week-5 Decision Tree measured 1.0000 accuracy under the original random 80/20 split. Under the time-aware split, it measured 0.9999 accuracy on the held-out period. However, the target was constructed using `gsc_impressions` and `gsc_clicks`, which were also model features. Therefore, these results should be treated as observed performance under this specific target construction and validation setup, rather than evidence of general predictive performance.

The model may provide directional decision-support, but stronger claims would require an independently defined outcome, leakage-free features, and validation across a broader time period.

## Self-check


- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/`